# ATML PA3 — Colab A100 runner
Select an **A100 GPU** runtime. This notebook runs the supplied implementations; it contains no fabricated results. Read `docs/ATML_PA3_Colab_Commands.pdf` for all commands, recovery, and evidence details. Run sections separately and back up often. **Do not Run all until you have provided the external inputs and reviewed the long-run cells.**

The guide is not the graded report. You must inspect labels, annotate the qualitative responses, interpret results, and write the report yourself.
Setup uses Python 3.11–3.13 in an isolated venv with matched CUDA PyTorch/torchvision. Every later shell cell explicitly activates it; Python subprocesses use PA3_PYTHON/PATH. Inside %%bash use python, never !python.


In [ ]:
# Upload the delivered atml-pa3-deliverable.zip, unless the source is already in /content/atml-pa3.
from pathlib import Path
from google.colab import files
import zipfile
root = Path('/content/atml-pa3')
if not root.exists():
    uploaded = files.upload()
    bundle = next(k for k in uploaded if k.endswith('.zip'))
    with zipfile.ZipFile(bundle) as archive:
        archive.extractall('/content')
%cd /content/atml-pa3


In [ ]:
# Install into a dedicated interpreter; keep Colab's kernel packages untouched.
import os, subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, 'scripts/setup_colab.py'], check=True)
PA3_ENV = Path('.venv-colab').resolve()
PA3_PYTHON = str(PA3_ENV / 'bin/python')
os.environ['PATH'] = str(PA3_ENV / 'bin') + os.pathsep + os.environ['PATH']


In [ ]:
# Verify the assignment interpreter, rather than importing Colab's kernel torch.
subprocess.run([PA3_PYTHON, '-m', 'atml_pa3.cli', 'doctor', '--require-cuda'], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
Path('/content/drive/MyDrive/ATML_PA3').mkdir(parents=True, exist_ok=True)


## Task 1 — shared preparation and baseline
Keep prepared data and the immutable model SHA across every method/session.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
pa3 task1 prepare
pa3 task1 evaluate --condition baseline


### Task 1 — full
This is a full 40,000-example run. Back up its recovery files before disconnecting.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
pa3 task1 train --method full
pa3 task1 evaluate --condition full_mid
pa3 task1 evaluate --condition full


### Task 1 — l2sp
This is a full 40,000-example run. Back up its recovery files before disconnecting.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
pa3 task1 train --method l2sp
pa3 task1 evaluate --condition l2sp


### Task 1 — lora4
This is a full 40,000-example run. Back up its recovery files before disconnecting.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
pa3 task1 train --method lora4
pa3 task1 evaluate --condition lora4


### Task 1 — lora16
This is a full 40,000-example run. Back up its recovery files before disconnecting.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
pa3 task1 train --method lora16
pa3 task1 evaluate --condition lora16


### Task 1 — oplora16
This is a full 40,000-example run. Back up its recovery files before disconnecting.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
pa3 task1 train --method oplora16
pa3 task1 evaluate --condition oplora16


### Task 1 — diagnostics and supplied prompts
The public-input helper fetches the exact prompt YAML linked inside the PDF and preserves its JSON system messages. Complete the resulting 42-row human annotation CSV yourself.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
python scripts/download_public_inputs.py
pa3 task1 diagnostics
pa3 task1 qualitative --prompts /content/atml-pa3/data/external/first_plot_questions.yaml
pa3 evidence


## Task 2 — data, features, curriculum, controls
Prepare the ten official annotation JSON files; inspect generated labels independently of predictions. Downloads fetch selected images only. This uses the Task 1 prepared Qwen revision.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
python scripts/download_annotations.py
pa3 task2 prepare --annotations data/external/coco/annotations
pa3 task2 cache --download


In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
pa3 task2 train --visual-tokens 196
pa3 task2 evaluate --visual-tokens 196
pa3 evidence


### Optional 49-token comparison
A complete independent 4,500-update run. Run this optional cell only if you choose to do the compression experiment.

In [ ]:
RUN_OPTIONAL_COMPRESSION = False
if RUN_OPTIONAL_COMPRESSION:
    subprocess.run(['pa3', 'task2', 'train', '--visual-tokens', '49'], check=True)
    subprocess.run(['pa3', 'task2', 'evaluate', '--visual-tokens', '49'], check=True)


## Task 3 — supplied rollout inputs
Set actual paths and the exact revision from the original README. Read the release's field definitions and adapt `configs/rollout_schema.json` before running. Problem text and MathVista metadata must come from released data, never generated substitutes.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
python scripts/download_public_inputs.py
python -m pip install gdown==5.2.0
gdown --folder 'https://drive.google.com/drive/folders/1utX-Gfi9vvaoi1mi0pzNpDGL1o1fAh23' -O /content/ATML-PA3-Data


In [ ]:
MATH_DIR = '/content/ATML-PA3-Data/LLM'
MATHVISTA_DIR = '/content/ATML-PA3-Data/VLM'
MATHVISTA_METADATA = '/content/atml-pa3/data/external/mathvista_testmini.json'
ROLLOUT_README = '/content/ATML-PA3-Data/README.MD'
LIVE_REVISION = 'REPLACE_WITH_EXACT_40_CHARACTER_SHA_FROM_README'


In [ ]:
subprocess.run(['pa3', 'task3', 'offline', '--math-dir', MATH_DIR,
    '--mathvista-dir', MATHVISTA_DIR, '--metadata', MATHVISTA_METADATA,
    '--readme', ROLLOUT_README, '--schema', 'configs/rollout_schema.json'], check=True)


In [ ]:
subprocess.run(['pa3', 'task3', 'live', '--revision', LIVE_REVISION,
    '--readme', ROLLOUT_README], check=True)


## Evidence and backup
The audit lists missing artifacts; it does not certify protocol compliance. Inspect counts, skipped steps, held-out splits, extraction/truncation, and raw outputs. Recovery commands are in the PDF guide.

In [ ]:
%%bash
set -euo pipefail
source .venv-colab/bin/activate
pa3 evidence
pa3 audit
python -m pip list --format=freeze > evidence/colab_environment.txt
mkdir -p /content/drive/MyDrive/ATML_PA3/checkpoints
cp -a evidence /content/drive/MyDrive/ATML_PA3/
cp -a checkpoints/. /content/drive/MyDrive/ATML_PA3/checkpoints/
cp -a data/prepared /content/drive/MyDrive/ATML_PA3/
